In [1]:
import time
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, datasets
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from torchmetrics.classification import MulticlassAccuracy
import torchvision.models as models

from PIL import Image

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {DEVICE}")

Using Device: cuda


In [2]:
class BrainTumorDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        super().__init__()        
        self.root_dir = root_dir
        self.transform = transform     
        self.classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        
        self.images = []
        for cls_name in self.classes:
            cls_path = os.path.join(root_dir, cls_name)
            for img_name in os.listdir(cls_path):
                self.images.append((os.path.join(cls_path, img_name), self.class_to_idx[cls_name]))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path, label = self.images[idx]
        image = Image.open(img_path).convert("RGB")       
        if self.transform:
            image = self.transform(image)
        return image, label

# Setup transforms and DataLoaders
data_path = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset'
mri_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

train_dataset = BrainTumorDataset(os.path.join(data_path, 'Training'), transform=mri_transforms)
test_dataset = BrainTumorDataset(os.path.join(data_path, 'Testing'), transform=mri_transforms)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

In [3]:
class CustomPrecisionModel(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()        
        # Original layers remain stored as FP32 weights
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2, 2)     
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2)   
        self.flatten = nn.Flatten()      
        flattened_size = 64 * 56 * 56
        self.fc1 = nn.Linear(flattened_size, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        print(f"Input image tensor type: {x.dtype}")       
        # Autocast for feature extraction. Conv layers use FP16. Print statements are just for verification, remove for training
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            x = self.conv1(x)
            print(f"Conv1 weight storage in: {self.conv1.weight.dtype} and activation op type: {x.dtype}")
            x = self.relu1(x)
            x = self.pool1(x)          
            x = self.conv2(x)
            print(f"Conv2 weight storage in: {self.conv2.weight.dtype} and activation op type: {x.dtype}")
            x = self.relu2(x)
            x = self.pool2(x)        
            x = self.flatten(x)          
        # Final dense layer connected to output layer so cast it to FP32
        x = x.to(torch.float32)
        print(f"Casting to FP32, verifying op : {x.dtype}")
        # Final layers FP32 for stability, disable autocasting
        with torch.amp.autocast(device_type='cuda', enabled=False):
            x = self.fc1(x)
            print(f"FC1 weight storage in: {self.fc1.weight.dtype} and activation op type: {x.dtype}")
            x = self.relu3(x)
            x = self.fc2(x)
            print(f"FC2 weight storage in: {self.fc2.weight.dtype} and activation op type: {x.dtype}")         
        return x

# Initialize Model and Optimizer normally
model = CustomPrecisionModel(num_classes=4).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [4]:
# GradScaler dynamically scales the loss for FP16 operations
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))
max_epochs = 2  #increase it to train. Make sure print statements in the previous cell is commented out!!
print("Starting Custom Mixed Precision Training")
# start_time = time.time()   ### uncomment if you want to time
##### Training loop, loop through all the epochs, if you have the previous prints on prepare to get flooded with prints!
for epoch in range(max_epochs):
    model.train()
    total_train_loss = 0.0
    correct_train = 0
    total_train = 0
    for batch_idx, (x, y) in enumerate(train_loader):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x)  # The forward pass method's precision controls what happens here
        loss = F.cross_entropy(logits, y)
        # Validation print for the first batch to verify our custom policy is working
        if epoch == 0 and batch_idx == 0 and DEVICE.type == 'cuda':
            print(f"Final logits dtype is {logits.dtype}. Should be torch.float32********")## VERY IMPORTANT!  
        # Scale loss and backpropagate
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        # Metrics
        total_train_loss += loss.item() * x.size(0)
        _, predicted = torch.max(logits, 1)
        total_train += y.size(0)
        correct_train += (predicted == y).sum().item()      
    epoch_train_loss = total_train_loss / len(train_dataset)
    epoch_train_acc = correct_train / total_train
    
    # Val loop
    model.eval()
    total_val_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)          
            logits = model(x)
            loss = F.cross_entropy(logits, y)                     
            total_val_loss += loss.item() * x.size(0)
            _, predicted = torch.max(logits, 1)
            total_val += y.size(0)
            correct_val += (predicted == y).sum().item()         
    epoch_val_loss = total_val_loss / len(test_dataset)
    epoch_val_acc = correct_val / total_val
    print(f"Epoch {epoch+1}/{max_epochs} -> "
          f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f} | "
          f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")
### Use this if you want to time
# end_time = time.time()
# print(f"Training time in seconds: {end_time - start_time:.4f}")

Starting Custom Mixed Precision Training
Input image tensor type: torch.float32
Conv1 weight storage in: torch.float32 and activation op type: torch.float16
Conv2 weight storage in: torch.float32 and activation op type: torch.float16
Casting to FP32, verifying op : torch.float32
FC1 weight storage in: torch.float32 and activation op type: torch.float32
FC2 weight storage in: torch.float32 and activation op type: torch.float32
Final logits dtype is torch.float32. Should be torch.float32********
Input image tensor type: torch.float32
Conv1 weight storage in: torch.float32 and activation op type: torch.float16
Conv2 weight storage in: torch.float32 and activation op type: torch.float16
Casting to FP32, verifying op : torch.float32
FC1 weight storage in: torch.float32 and activation op type: torch.float32
FC2 weight storage in: torch.float32 and activation op type: torch.float32
Input image tensor type: torch.float32
Conv1 weight storage in: torch.float32 and activation op type: torch.floa